# Exercise 1: Blockchain Structure and Tamper Detection

In [ ]:
import hashlib
import pickle
import time

def hash_data(data):
    return hashlib.sha256(data.encode()).hexdigest()

def create_merkle_root(transactions):
    if len(transactions) == 0:
        return ""

    hashes = [hash_data(tx) for tx in transactions]

    while len(hashes) > 1:
        if len(hashes) % 2 != 0:
            hashes.append(hashes[-1])

        next_level = []

        for i in range(0, len(hashes), 2):
            combined = hashes[i] + hashes[i + 1]
            next_level.append(hash_data(combined))

        hashes = next_level

    return hashes[0]

class Block:
    def __init__(self, height, transactions, previous_hash):
        self.height = height
        self.timestamp = int(time.time())
        self.previous_hash = previous_hash
        self.transactions = transactions
        self.merkle_root = create_merkle_root(transactions)
        self.hash = self.create_hash()

    def create_hash(self):
        block_data = {
            "height": self.height,
            "timestamp": self.timestamp,
            "previous_hash": self.previous_hash,
            "merkle_root": self.merkle_root
        }

        block_bytes = pickle.dumps(block_data)
        return hashlib.sha256(block_bytes).hexdigest()

class Blockchain:
    def __init__(self):
        genesis_block = Block(
            1,
            ["Genesis Block"],
            "Genesis Block"
        )
        self.chain = [genesis_block]

    def add_block(self, transactions):
        previous_block = self.chain[-1]
        new_block = Block(
            len(self.chain) + 1,
            transactions,
            previous_block.hash
        )
        self.chain.append(new_block)

    def validate_chain(self):
        for i in range(len(self.chain)):
            current_block = self.chain[i]

            recalculated_merkle_root = create_merkle_root(
                current_block.transactions
            )

            if current_block.merkle_root != recalculated_merkle_root:
                print("Chain Invalid")
                print("First invalid block:", current_block.height)
                print("Reason: Transaction data does not match the Merkle Root")
                print("Stored Merkle Root:", current_block.merkle_root)
                print("Recalculated Merkle Root:", recalculated_merkle_root)
                return False

            if current_block.hash != current_block.create_hash():
                print("Chain Invalid")
                print("First invalid block:", current_block.height)
                print("Reason: Stored block hash is incorrect")
                print("Stored block hash:", current_block.hash)
                print("Recalculated block hash:", current_block.create_hash())
                return False

            if i > 0:
                previous_block = self.chain[i - 1]
                if current_block.previous_hash != previous_block.hash:
                    print("Chain Invalid")
                    print("First invalid block:", current_block.height)
                    print("Reason: Previous block hash does not match")
                    print("Stored previous hash:", current_block.previous_hash)
                    print("Expected previous hash:", previous_block.hash)
                    return False

        print("Chain Valid")
        return True

    def print_chain(self):
        for block in self.chain:
            print("Block Height:", block.height)
            print("Timestamp:", block.timestamp)
            print("Previous Hash:", block.previous_hash)
            print("Transactions:", block.transactions)
            print("Merkle Root:", block.merkle_root)
            print("Block Hash:", block.hash)
            print()

blockchain = Blockchain()

blockchain.add_block(["account1 -> account2: 100", "account2 -> account3: 50"])
blockchain.add_block(["account3 -> account4: 25", "account4 -> account5: 10"])
blockchain.add_block(["account5 -> account1: 20", "account1 -> account3: 5"])
blockchain.add_block(["account2 -> account4: 40", "account4 -> account1: 15"])
blockchain.add_block(["account3 -> account5: 30", "account5 -> account2: 8"])
blockchain.add_block(["account1 -> account4: 12", "account4 -> account3: 6"])
blockchain.add_block(["account2 -> account5: 18", "account5 -> account1: 9"])
blockchain.add_block(["account3 -> account1: 22", "account1 -> account2: 11"])
blockchain.add_block(["account4 -> account2: 16", "account2 -> account5: 4"])

print("Number of blocks:", len(blockchain.chain))
print()
print("BLOCKCHAIN BEFORE MODIFICATION")
print()
blockchain.print_chain()
print("VALIDATION BEFORE MODIFICATION")
blockchain.validate_chain()

blockchain.chain[4].transactions[0] = "account2 -> account4: 4000"

print()
print("BLOCKCHAIN AFTER MODIFICATION")
print()
blockchain.print_chain()
print("VALIDATION AFTER MODIFICATION")
blockchain.validate_chain()

Number of blocks: 10

BLOCKCHAIN BEFORE MODIFICATION

Block Height: 1
Timestamp: 1790568423
Previous Hash: Genesis Block
Transactions: ['Genesis Block']
Merkle Root: 89eb0ac031a63d2421cd05a2fbe41f3ea35f5c3712ca839cbf6b85c4ee07b7a3
Block Hash: 4ed90c197e316ac0dd5d0ff25ba60b8c2946b525aba3b767e523303867348032

Block Height: 2
Timestamp: 1790568423
Previous Hash: 4ed90c197e316ac0dd5d0ff25ba60b8c2946b525aba3b767e523303867348032
Transactions: ['account1 -> account2: 100', 'account2 -> account3: 50']
Merkle Root: 06b793ad5cb4d01b1a86344110f89cbc2ab6022e4ffa7f35bf6455a657ada006
Block Hash: fc6e560fa9adb971af8c016eebb34d8531eea85670b289a22683aa77071164db

Block Height: 3
Timestamp: 1790568423
Previous Hash: fc6e560fa9adb971af8c016eebb34d8531eea85670b289a22683aa77071164db
Transactions: ['account3 -> account4: 25', 'account4 -> account5: 10']
Merkle Root: 49ffc7838fb5439ebdde5e8379f4818513bb07b55de29a4480652de88fa83955
Block Hash: f12561df688b68a17ef2906bc186f93c9fa45357958600273d88184e94a20a3a



False